In [1]:
import pandas as pd
import glob
import os

print("Isolating and formatting Central Park weather data...")

# Load and clean the weather data
weather_df = pd.read_csv('noaa_weather_2025.csv')
central_park_weather = weather_df[weather_df['STATION'] == 'USW00094728'].copy()
central_park_weather = central_park_weather[['DATE', 'TMAX', 'TMIN', 'PRCP', 'AWND']]
central_park_weather['DATE'] = pd.to_datetime(central_park_weather['DATE']).dt.normalize()

print("Weather data is ready for merging:")
display(central_park_weather.head(3))

Isolating and formatting Central Park weather data...
Weather data is ready for merging:


,DATE,TMAX,TMIN,PRCP,AWND
0,2025-01-01,51,39,0.0,8.28
1,2025-01-02,43,33,0.0,9.62
2,2025-01-03,39,31,0.0,7.83


In [2]:
# 1. Find all files, ignoring Jersey City and hidden Mac files
all_files = [f for f in glob.glob('*citibike-tripdata*.csv') if 'JC' not in f and not f.startswith('._')]
all_files.sort() 

print(f"Found {len(all_files)} NYC files! Starting full-year processing...")

# The optimized raw file we are building for your team
raw_output_file = 'df_merged_2025_FULL_YEAR.csv'
if os.path.exists(raw_output_file):
    os.remove(raw_output_file)

# We are using your teammate's exact logic here: NO lat/lng columns!
needed_cols = [
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "member_casual"
]

daily_summaries = []
first_chunk = True

# Process file by file (exactly like your teammate did, but saving locally)
for i, file in enumerate(all_files, start=1):
    print(f"[{i}/{len(all_files)}] Processing: {file}")
    
    # Read the file using only the necessary columns to save memory
    temp = pd.read_csv(file, usecols=needed_cols, low_memory=False)
    
    # Format the date to match the weather dataset
    temp['started_at'] = pd.to_datetime(temp['started_at'])
    temp['DATE'] = temp['started_at'].dt.normalize()
    
    # Merge weather onto the raw rides
    temp = pd.merge(temp, central_park_weather, on='DATE', how='inner')
    
    # SAVE DIRECTLY TO YOUR MAC HARD DRIVE
    temp.to_csv(raw_output_file, mode='a', header=first_chunk, index=False)
    first_chunk = False
    
    # AGGREGATE FOR THE ML MODEL
    daily_chunk = temp.groupby(['DATE', 'TMAX', 'TMIN', 'PRCP', 'AWND']).size().reset_index(name='Total_Daily_Rides')
    daily_summaries.append(daily_chunk)

print(f"\nSUCCESS! The full-year raw dataset is safely saved as: '{raw_output_file}'")

Found 53 NYC files! Starting full-year processing...
[1/53] Processing: 202501-citibike-tripdata_1.csv
[2/53] Processing: 202501-citibike-tripdata_2.csv
[3/53] Processing: 202501-citibike-tripdata_3.csv
[4/53] Processing: 202502-citibike-tripdata_1.csv
[5/53] Processing: 202502-citibike-tripdata_2.csv
[6/53] Processing: 202502-citibike-tripdata_3.csv
[7/53] Processing: 202503-citibike-tripdata_1.csv
[8/53] Processing: 202503-citibike-tripdata_2.csv
[9/53] Processing: 202503-citibike-tripdata_3.csv
[10/53] Processing: 202503-citibike-tripdata_4.csv
[11/53] Processing: 202504-citibike-tripdata_1.csv
[12/53] Processing: 202504-citibike-tripdata_2.csv
[13/53] Processing: 202504-citibike-tripdata_3.csv
[14/53] Processing: 202504-citibike-tripdata_4.csv
[15/53] Processing: 202505-citibike-tripdata_1.csv
[16/53] Processing: 202505-citibike-tripdata_2.csv
[17/53] Processing: 202505-citibike-tripdata_3.csv
[18/53] Processing: 202505-citibike-tripdata_4.csv
[19/53] Processing: 202505-citibike-tr

In [5]:
import pandas as pd

print("Loading the massive full-year dataset... this might take a minute!")

# Open the RAW file that has every single ride merged with weather
df_full_raw = pd.read_csv('df_merged_2025_FULL_YEAR.csv')

print(f"Total rides in this dataset: {len(df_full_raw):,}")
print("\nHere is your unaggregated data with all columns together:")

# Look at the first 5 rows
display(df_full_raw.head())

Loading the massive full-year dataset... this might take a minute!
Total rides in this dataset: 45,770,895

Here is your unaggregated data with all columns together:


,started_at,ended_at,start_station_name,end_station_name,member_casual,DATE,TMAX,TMIN,PRCP,AWND
0,2025-01-01 14:52:26.542,2025-01-01 14:59:53.427,W 20 St & 7 Ave,E 10 St & 2 Ave,casual,2025-01-01,51,39,0.00,8.28
1,2025-01-10 05:03:13.646,2025-01-10 05:13:13.331,W 20 St & 7 Ave,E 25 St & 1 Ave,member,2025-01-10,39,27,0.00,8.05
2,2025-01-13 13:40:17.630,2025-01-13 13:47:05.817,St James Pl & Oliver St,Fulton St & William St,member,2025-01-13,42,33,0.00,5.37
3,2025-01-10 08:29:16.996,2025-01-10 08:34:49.360,St James Pl & Oliver St,Fulton St & William St,member,2025-01-10,39,27,0.00,8.05
4,2025-01-11 18:59:48.427,2025-01-11 19:13:21.292,E 33 St & 1 Ave,Allen St & Rivington St,member,2025-01-11,36,28,0.02,7.16


In [7]:
import pandas as pd

print("Checking the massive raw file... (Loading just the 'DATE' column to save time!)")

# Read ONLY the uppercase 'DATE' column from the 6GB file
df_dates = pd.read_csv('df_merged_2025_FULL_YEAR.csv', usecols=['DATE'])

# Convert to datetime so we can extract math/stats from it
df_dates['DATE'] = pd.to_datetime(df_dates['DATE'])

print("\n--- Raw Dataset Verification ---")
print(f"First day of data: {df_dates['DATE'].min().date()}")
print(f"Last day of data:  {df_dates['DATE'].max().date()}")
print(f"Total unique days recorded: {df_dates['DATE'].nunique()} days (Should be 365)")

print("\n--- Rides per Month Breakdown ---")
# Extract the month and count how many rides happened each month
df_dates['Month'] = df_dates['DATE'].dt.month
rides_per_month = df_dates['Month'].value_counts().sort_index()

# Display the ride counts beautifully
for month, count in rides_per_month.items():
    print(f"Month {month:02d}: {count:>10,} rides")
    
if df_dates['DATE'].nunique() == 365:
    print("\n✅ SUCCESS: Your raw file perfectly covers the entire 2025 year!")
else:
    print("\n⚠️ WARNING: Some days might be missing.")

Checking the massive raw file... (Loading just the 'DATE' column to save time!)

--- Raw Dataset Verification ---
First day of data: 2025-01-01
Last day of data:  2025-12-31
Total unique days recorded: 365 days (Should be 365)

--- Rides per Month Breakdown ---
Month 01:  2,124,541 rides
Month 02:  2,031,394 rides
Month 03:  3,167,962 rides
Month 04:  3,725,037 rides
Month 05:  4,325,897 rides
Month 06:  4,759,239 rides
Month 07:  4,987,563 rides
Month 08:  5,122,174 rides
Month 09:  5,287,155 rides
Month 10:  4,732,358 rides
Month 11:  3,416,092 rides
Month 12:  2,091,483 rides

✅ SUCCESS: Your raw file perfectly covers the entire 2025 year!
